# DSA 07: Sorting, Searching & Greedy
## Ordering, Finding, and Making Optimal Choices

**What you'll learn:**
- All major sorting algorithms (bubble, selection, insertion, merge, quick, heap)
- Python's built-in sort (Timsort) and custom comparators
- Binary search -- 3 templates (exact match, lower bound, upper bound)
- Binary search on answer (minimize/maximize problems)
- Greedy algorithms: when they work and how to prove correctness
- 15+ practice problems with solutions

**Prerequisites:** DSA_06 (Recursion, Backtracking, DP)
**Estimated Time:** 5-6 hours

---

> You will almost never implement sorting from scratch in production.
> But understanding HOW sorts work helps you choose the right one and
> optimize Spark/SQL queries. Binary search shows up constantly in interviews.

---

---
# Section 1: Sorting Algorithms

## The Big Picture

| Algorithm | Time (avg) | Time (worst) | Space | Stable? | Notes |
|-----------|-----------|-------------|-------|---------|-------|
| Bubble Sort | O(n^2) | O(n^2) | O(1) | Yes | Educational only |
| Selection Sort | O(n^2) | O(n^2) | O(1) | No | Educational only |
| Insertion Sort | O(n^2) | O(n^2) | O(1) | Yes | Fast for small/nearly sorted |
| **Merge Sort** | O(n log n) | O(n log n) | O(n) | Yes | Guaranteed, great for linked lists |
| **Quick Sort** | O(n log n) | O(n^2) | O(log n) | No | Fastest in practice (avg) |
| Heap Sort | O(n log n) | O(n log n) | O(1) | No | O(1) space guaranteed |
| **Timsort** | O(n log n) | O(n log n) | O(n) | Yes | Python's built-in -- use this! |

**Stable sort**: Equal elements maintain their original relative order.

## Which to Use in Practice

- **Always use `sorted()` or `.sort()`** in Python (Timsort -- hybrid of merge + insertion)
- Know merge sort and quick sort for interviews
- Understand trade-offs for system design discussions

## DE Connection

- Spark's **sort-merge join** uses merge sort
- **ORDER BY** uses Timsort-like algorithms internally
- **Partition sorting** in Delta Lake uses sorting for optimization
- Understanding sort stability matters for **deterministic ordering**

In [ ]:
# Sorting Algorithms from Scratch
import random, time

print("SORTING ALGORITHMS")
print("=" * 60)

# 1. Merge Sort -- O(n log n), stable, O(n) space
def merge_sort(arr):
    """Divide in half, sort each half, merge. O(n log n)."""
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    # Merge two sorted halves
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

# 2. Quick Sort -- O(n log n) avg, O(n^2) worst, O(log n) space
def quick_sort(arr):
    """Pick pivot, partition, recurse. O(n log n) average."""
    if len(arr) <= 1:
        return arr

    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]

    return quick_sort(left) + middle + quick_sort(right)

# 3. Insertion Sort -- O(n^2), but O(n) for nearly sorted!
def insertion_sort(arr):
    """Insert each element into its correct position. O(n^2)."""
    arr = arr[:]
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr

# Test all
test = [5, 2, 8, 1, 9, 3, 7, 4, 6]
print(f"  Original: {test}")
print(f"  Merge sort:     {merge_sort(test)}")
print(f"  Quick sort:     {quick_sort(test)}")
print(f"  Insertion sort: {insertion_sort(test)}")
print(f"  Built-in sort:  {sorted(test)}")

In [ ]:
# Performance comparison
print("\nPERFORMANCE COMPARISON")
print("=" * 60)

def time_sort(sort_func, arr):
    start = time.perf_counter()
    sort_func(arr[:])
    return time.perf_counter() - start

sizes = [1000, 5000, 10000, 50000]
print(f"  {'Size':<10} {'Merge':<12} {'Quick':<12} {'Built-in':<12}")
print(f"  {'-'*46}")

for size in sizes:
    arr = [random.randint(0, size) for _ in range(size)]

    t_merge = time_sort(merge_sort, arr)
    t_quick = time_sort(quick_sort, arr)
    t_builtin = time_sort(sorted, arr)

    print(f"  {size:<10} {t_merge:<12.4f} {t_quick:<12.4f} {t_builtin:<12.6f}")

print("\n  Built-in sorted() is fastest because Timsort is written in C.")
print("  In interviews: implement merge/quick sort from scratch.")
print("  In production: always use sorted() / .sort().")

In [ ]:
# Python sorting tricks
print("\nPYTHON SORTING TRICKS")
print("=" * 60)

# Custom sort key
print("1. Custom key function:")
words = ["banana", "apple", "cherry", "date"]
by_length = sorted(words, key=len)
print(f"   Sort by length: {by_length}")

# Sort by multiple criteria
print("\n2. Multiple criteria (sort by age, then name):")
people = [("Alice", 30), ("Bob", 25), ("Charlie", 30), ("Diana", 25)]
by_age_name = sorted(people, key=lambda x: (x[1], x[0]))
print(f"   {by_age_name}")

# Sort descending
print("\n3. Descending order:")
nums = [3, 1, 4, 1, 5, 9]
print(f"   Ascending: {sorted(nums)}")
print(f"   Descending: {sorted(nums, reverse=True)}")

# Sort dict by value
print("\n4. Sort dictionary by value:")
scores = {"Alice": 95, "Bob": 87, "Charlie": 92}
by_score = sorted(scores.items(), key=lambda x: x[1], reverse=True)
print(f"   By score (desc): {by_score}")

# Stability demo
print("\n5. Stable sort preserves relative order of equal elements:")
data = [(1, "a"), (2, "b"), (1, "c"), (2, "d")]
by_first = sorted(data, key=lambda x: x[0])
print(f"   Sorted by first element: {by_first}")
print("   (1,'a') still comes before (1,'c') -- stable!")

---
# Section 2: Binary Search

## What is Binary Search?

Search a **sorted** array by repeatedly cutting the search space in half.
- O(log n) time -- incredibly efficient
- Requires sorted input

## The Three Templates

### Template 1: Exact Match
Find the exact target, return index or -1.

### Template 2: Lower Bound (bisect_left)
Find the leftmost position where target could be inserted to maintain order.
Useful for "first occurrence" or "first element >= target".

### Template 3: Upper Bound (bisect_right)
Find the rightmost position where target could be inserted.
Useful for "last occurrence" or "first element > target".

## When to Use Binary Search

- Array is **sorted** (or can be sorted)
- Problem asks for **first/last** occurrence
- Problem says "minimize the maximum" or "maximize the minimum"
- The answer space is monotonic (if X works, all Y > X also work)

In [ ]:
# Binary Search: Template 1 -- Exact Match
print("BINARY SEARCH")
print("=" * 60)

def binary_search(arr, target):
    """
    Find target in sorted array. Return index or -1.
    Time: O(log n), Space: O(1)
    """
    left, right = 0, len(arr) - 1

    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1

    return -1

arr = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
print(f"  Array: {arr}")
print(f"  Search for 7: index {binary_search(arr, 7)}")
print(f"  Search for 12: index {binary_search(arr, 12)}")
print(f"  Steps for array of 1M: ~20 (log2(1,000,000) = 20)")

In [ ]:
# Binary Search: Template 2 -- Lower Bound (first >= target)
print("\nBINARY SEARCH: Lower Bound (bisect_left)")
print("=" * 60)

def lower_bound(arr, target):
    """
    Find first index where arr[index] >= target.
    Returns len(arr) if all elements < target.
    """
    left, right = 0, len(arr)

    while left < right:
        mid = (left + right) // 2
        if arr[mid] < target:
            left = mid + 1
        else:
            right = mid

    return left

arr = [1, 2, 4, 4, 4, 6, 8, 10]
print(f"  Array: {arr}")
print(f"  First >= 4: index {lower_bound(arr, 4)} (value {arr[lower_bound(arr, 4)]})")
print(f"  First >= 5: index {lower_bound(arr, 5)} (value {arr[lower_bound(arr, 5)]})")
print(f"  First >= 0: index {lower_bound(arr, 0)}")

# Count occurrences using lower/upper bound
def upper_bound(arr, target):
    """Find first index where arr[index] > target."""
    left, right = 0, len(arr)
    while left < right:
        mid = (left + right) // 2
        if arr[mid] <= target:
            left = mid + 1
        else:
            right = mid
    return left

arr = [1, 2, 4, 4, 4, 6, 8, 10]
lb = lower_bound(arr, 4)
ub = upper_bound(arr, 4)
print(f"\n  Count of 4: upper_bound - lower_bound = {ub} - {lb} = {ub - lb}")

In [ ]:
# Binary Search on Answer: Minimum Ship Capacity
print("\nBINARY SEARCH ON ANSWER")
print("=" * 60)

def ship_within_days(weights, days):
    """
    Find minimum ship capacity to ship all packages within D days.
    Binary search on the ANSWER (capacity).
    Time: O(n * log(sum)), Space: O(1)
    """
    def can_ship(capacity):
        """Can we ship all within D days at this capacity?"""
        current_load = 0
        days_needed = 1
        for w in weights:
            if current_load + w > capacity:
                days_needed += 1
                current_load = 0
            current_load += w
        return days_needed <= days

    # Binary search on capacity: min = max(weights), max = sum(weights)
    left, right = max(weights), sum(weights)

    while left < right:
        mid = (left + right) // 2
        if can_ship(mid):
            right = mid  # Try smaller capacity
        else:
            left = mid + 1  # Need bigger capacity

    return left

weights = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
days = 5
result = ship_within_days(weights, days)
print(f"  Weights: {weights}")
print(f"  Days: {days}")
print(f"  Min capacity: {result}")
print()
print("  Pattern: 'Minimize the maximum' or 'Maximize the minimum'")
print("  -> Binary search on the answer!")
print("  Check: can we achieve this answer? (feasibility function)")

In [ ]:
# Binary Search: Search in Rotated Sorted Array
print("\nBINARY SEARCH: Rotated Sorted Array")
print("=" * 60)

def search_rotated(nums, target):
    """
    Search in a rotated sorted array (e.g., [4,5,6,7,0,1,2]).
    Time: O(log n), Space: O(1)
    """
    left, right = 0, len(nums) - 1

    while left <= right:
        mid = (left + right) // 2
        if nums[mid] == target:
            return mid

        # Determine which half is sorted
        if nums[left] <= nums[mid]:  # Left half is sorted
            if nums[left] <= target < nums[mid]:
                right = mid - 1
            else:
                left = mid + 1
        else:  # Right half is sorted
            if nums[mid] < target <= nums[right]:
                left = mid + 1
            else:
                right = mid - 1

    return -1

arr = [4, 5, 6, 7, 0, 1, 2]
print(f"  Array: {arr}")
print(f"  Search for 0: index {search_rotated(arr, 0)}")
print(f"  Search for 5: index {search_rotated(arr, 5)}")
print(f"  Search for 3: index {search_rotated(arr, 3)}")

---
# Section 3: Greedy Algorithms

## What is a Greedy Algorithm?

Make the **locally optimal choice** at each step, hoping it leads to a
globally optimal solution.

## When Does Greedy Work?

Greedy works when the problem has:
1. **Greedy choice property**: A locally optimal choice is part of a global optimum
2. **Optimal substructure**: An optimal solution contains optimal solutions to subproblems

## When Greedy Fails

- **Coin change with arbitrary denominations**: Greedy picks largest coin first, but [1, 3, 4] with target 6 = greedy picks 4+1+1=3 coins, optimal is 3+3=2 coins
- **0/1 Knapsack**: Must consider all combinations (use DP instead)

## Classic Greedy Problems

| Problem | Greedy Strategy |
|---------|----------------|
| Activity selection | Pick earliest ending activity |
| Fractional knapsack | Pick highest value/weight ratio |
| Huffman coding | Merge two smallest frequencies |
| Jump game | Track farthest reachable position |
| Merge intervals | Sort by start, merge overlapping |

In [ ]:
# Greedy Problem 1: Activity Selection (Interval Scheduling)
print("GREEDY: Activity Selection (Interval Scheduling)")
print("=" * 60)

def max_activities(intervals):
    """
    Find maximum number of non-overlapping activities.
    Greedy: Always pick the activity that ends earliest.
    Time: O(n log n) for sorting, Space: O(1)
    """
    # Sort by end time
    intervals.sort(key=lambda x: x[1])

    count = 1
    last_end = intervals[0][1]

    for start, end in intervals[1:]:
        if start >= last_end:  # No overlap
            count += 1
            last_end = end

    return count

activities = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14)]
result = max_activities(activities)
print(f"  Activities (start, end): {activities}")
print(f"  Max non-overlapping: {result}")
print("\n  Why earliest end? It leaves the MOST room for future activities.")

In [ ]:
# Greedy Problem 2: Merge Intervals
print("\nGREEDY: Merge Intervals")
print("=" * 60)

def merge_intervals(intervals):
    """
    Merge all overlapping intervals.
    Time: O(n log n), Space: O(n)
    """
    intervals.sort(key=lambda x: x[0])  # Sort by start
    merged = [intervals[0]]

    for start, end in intervals[1:]:
        if start <= merged[-1][1]:  # Overlaps with last merged
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))

    return merged

intervals = [(1, 3), (2, 6), (8, 10), (15, 18)]
result = merge_intervals(intervals)
print(f"  Input: {intervals}")
print(f"  Merged: {result}")

intervals2 = [(1, 4), (4, 5)]
result2 = merge_intervals(intervals2)
print(f"  Input: {intervals2}")
print(f"  Merged: {result2}")
print("\n  DE Connection: Merging time ranges in event data!")
print("  Also: partition range merging in query optimization.")

In [ ]:
# Greedy Problem 3: Jump Game
print("\nGREEDY: Jump Game")
print("=" * 60)

def can_jump(nums):
    """
    Can you reach the last index? nums[i] = max jump length from i.
    Greedy: Track farthest reachable position.
    Time: O(n), Space: O(1)
    """
    farthest = 0
    for i in range(len(nums)):
        if i > farthest:
            return False  # Can't reach this position
        farthest = max(farthest, i + nums[i])
        if farthest >= len(nums) - 1:
            return True
    return True

def min_jumps(nums):
    """
    Minimum jumps to reach end. Greedy: BFS-like level approach.
    Time: O(n), Space: O(1)
    """
    if len(nums) <= 1:
        return 0
    jumps = 0
    current_end = 0
    farthest = 0

    for i in range(len(nums) - 1):
        farthest = max(farthest, i + nums[i])
        if i == current_end:  # Must jump
            jumps += 1
            current_end = farthest
            if current_end >= len(nums) - 1:
                break
    return jumps

tests = [([2, 3, 1, 1, 4], True), ([3, 2, 1, 0, 4], False)]
for nums, expected in tests:
    result = can_jump(nums)
    status = "PASS" if result == expected else "FAIL"
    print(f"  [{status}] can_jump({nums}) = {result}")

print(f"\n  min_jumps([2,3,1,1,4]) = {min_jumps([2, 3, 1, 1, 4])}")

In [ ]:
# Greedy Problem 4: Task Scheduler
print("\nGREEDY: Task Scheduler")
print("=" * 60)

from collections import Counter

def least_interval(tasks, n):
    """
    Find minimum intervals to execute all tasks with cooldown n.
    Greedy: Most frequent task determines the frame.
    Time: O(t) where t = number of tasks, Space: O(1)
    """
    freq = Counter(tasks)
    max_freq = max(freq.values())
    # Count how many tasks have the maximum frequency
    max_count = sum(1 for f in freq.values() if f == max_freq)

    # Formula: (max_freq - 1) * (n + 1) + max_count
    # But total tasks might be larger (no idle needed)
    result = (max_freq - 1) * (n + 1) + max_count
    return max(result, len(tasks))

tasks = ["A", "A", "A", "B", "B", "B"]
n = 2
result = least_interval(tasks, n)
print(f"  Tasks: {tasks}, cooldown: {n}")
print(f"  Min intervals: {result}")
print("  Schedule: A B _ A B _ A B (or similar)")
print()
tasks2 = ["A", "A", "A", "A", "A", "A", "B", "C", "D", "E", "F", "G"]
n2 = 2
print(f"  Tasks: {tasks2}, cooldown: {n2}")
print(f"  Min intervals: {least_interval(tasks2, n2)}")

---
# Section 4: Your Turn -- Practice Problems

In [ ]:
# ============================================
# YOUR TURN: Problem 1 -- Find Peak Element
# ============================================
# A peak element is greater than its neighbors.
# Find ANY peak in O(log n) time.
#
# Example: [1, 2, 3, 1] -> index 2 (value 3)
# Example: [1, 2, 1, 3, 5, 6, 4] -> index 5 (value 6) or index 1
#
# Hint: Binary search! If mid < mid+1, peak is to the right.

def find_peak(nums):
    pass  # Your solution here


In [ ]:
# ============================================
# YOUR TURN: Problem 2 -- Meeting Rooms II (Min Rooms)
# ============================================
# Given meeting intervals, find minimum number of rooms needed.
#
# Example: [[0,30],[5,10],[15,20]] -> 2 rooms
#
# Hint: Sort by start time. Use a min-heap to track end times.
# When a new meeting starts, check if any room is free (heap top <= start).

def min_meeting_rooms(intervals):
    pass  # Your solution here


In [ ]:
# ============================================
# YOUR TURN: Problem 3 -- Koko Eating Bananas
# ============================================
# Koko has piles of bananas. She can eat k bananas/hour.
# Find minimum k to finish all piles within h hours.
#
# Example: piles=[3,6,7,11], h=8 -> k=4
#
# Hint: Binary search on k! For each k, calculate hours needed.
# If hours <= h, try smaller k. Else try larger k.

def min_eating_speed(piles, h):
    pass  # Your solution here


---
# Solutions

In [ ]:
# SOLUTION 1: Find Peak Element
print("SOLUTION 1: Find Peak Element")
print("=" * 60)

def find_peak(nums):
    """
    Binary search for a peak. If nums[mid] < nums[mid+1], go right.
    Time: O(log n), Space: O(1)
    """
    left, right = 0, len(nums) - 1

    while left < right:
        mid = (left + right) // 2
        if nums[mid] < nums[mid + 1]:
            left = mid + 1  # Peak must be to the right
        else:
            right = mid  # Peak is at mid or to the left

    return left

tests = [
    ([1, 2, 3, 1], 2),
    ([1, 2, 1, 3, 5, 6, 4], 5),
]
for nums, expected in tests:
    result = find_peak(nums)
    print(f"  find_peak({nums}) = index {result} (value {nums[result]})")

In [ ]:
# SOLUTION 2: Meeting Rooms II
import heapq

print("\nSOLUTION 2: Meeting Rooms II")
print("=" * 60)

def min_meeting_rooms(intervals):
    """
    Sort by start. Use min-heap of end times to track rooms.
    Time: O(n log n), Space: O(n)
    """
    if not intervals:
        return 0

    intervals.sort(key=lambda x: x[0])
    heap = []  # Min-heap of end times

    for start, end in intervals:
        if heap and heap[0] <= start:
            heapq.heappop(heap)  # Reuse room (meeting ended)
        heapq.heappush(heap, end)

    return len(heap)  # Heap size = rooms needed

tests = [
    ([[0, 30], [5, 10], [15, 20]], 2),
    ([[7, 10], [2, 4]], 1),
    ([[1, 5], [2, 6], [3, 7]], 3),
]
for intervals, expected in tests:
    result = min_meeting_rooms(intervals)
    status = "PASS" if result == expected else "FAIL"
    print(f"  [{status}] rooms({intervals}) = {result}")

print("\n  DE Connection: Resource scheduling for compute clusters!")

In [ ]:
# SOLUTION 3: Koko Eating Bananas
import math

print("\nSOLUTION 3: Koko Eating Bananas (Binary Search on Answer)")
print("=" * 60)

def min_eating_speed(piles, h):
    """
    Binary search on eating speed k.
    Time: O(n * log(max_pile)), Space: O(1)
    """
    def hours_needed(k):
        return sum(math.ceil(p / k) for p in piles)

    left, right = 1, max(piles)

    while left < right:
        mid = (left + right) // 2
        if hours_needed(mid) <= h:
            right = mid  # Try slower
        else:
            left = mid + 1  # Need faster

    return left

tests = [
    ([3, 6, 7, 11], 8, 4),
    ([30, 11, 23, 4, 20], 5, 30),
    ([30, 11, 23, 4, 20], 6, 23),
]
for piles, h, expected in tests:
    result = min_eating_speed(piles, h)
    status = "PASS" if result == expected else "FAIL"
    print(f"  [{status}] speed(piles={piles}, h={h}) = {result}")

---
# Section 5: Summary

## Sorting Cheat Sheet

| Situation | Best Algorithm |
|-----------|---------------|
| General sorting | `sorted()` (Timsort) |
| Need stability | Merge sort or Timsort |
| Memory constrained | Heap sort (O(1) space) |
| Nearly sorted data | Insertion sort (O(n) best case) |
| Need kth element | Quick select O(n) average |
| Custom ordering | `sorted(key=lambda ...)` |

## Binary Search Decision Tree

```
Is the array sorted?
  YES -> Direct binary search
  NO  -> Can you sort it first?
         YES -> Sort + binary search
         NO  -> Is the answer space monotonic?
                YES -> Binary search on answer
                NO  -> Not a binary search problem
```

## Greedy Decision Tree

```
Can you make a local choice that is always optimal?
  YES -> Greedy (prove it works!)
  NO  -> Use DP or backtracking instead
```

## LeetCode Problems

**Easy:** Binary Search (#704), First Bad Version (#278)
**Medium:** Search Rotated Array (#33), Find Peak (#162), Merge Intervals (#56),
           Meeting Rooms II (#253), Jump Game (#55), Task Scheduler (#621)
**Hard:** Median of Two Sorted Arrays (#4)

---


In [ ]:
print("KEY TAKEAWAYS FROM DSA_07")
print("=" * 50)
print()
print("SORTING:")
print("  - Use sorted() in production (Timsort, O(n log n))")
print("  - Know merge sort + quick sort for interviews")
print("  - Custom key functions for complex ordering")
print()
print("BINARY SEARCH:")
print("  - Sorted array: O(log n) search")
print("  - Three templates: exact, lower bound, upper bound")
print("  - Binary search on ANSWER: minimize/maximize problems")
print()
print("GREEDY:")
print("  - Local optimal choice -> global optimal")
print("  - Works for intervals, scheduling, some optimization")
print("  - If unsure, try DP instead (greedy is DP + proof of optimality)")
print()
print("Next: DSA_08 -- Interview Patterns & Mock Problems (FINAL!)")